# CQADupStack Non-Federated Diagnostic

**Purpose**: FiQA (D1–D5) was already tested and failed — `all-MiniLM-L6-v2` is at ceiling on financial text (pre-train MRR 0.168, training loss ~0.47). This run tests CQADupStack sub-forums whose niche technical language should give the model genuine headroom.

**Applies to**: Both QA-FedAvg (Phase 1A) and DAS-FedAvg (Phase 1B) — same model, same loss, same dataset.

### Variants run by this notebook

| ID | Dataset | LR | Max Train Pairs | Rationale |
|----|---------|-----|-----------------|----------|
| C1 | cqadupstack/android | 2e-6 | 4000 | Proven QA-FedAvg LR |
| C2 | cqadupstack/android | 5e-7 | 4000 | Proven DAS-FedAvg LR |
| C3 | cqadupstack/android | 1e-6 | 4000 | Midpoint LR |
| C4 | cqadupstack/tex | 2e-6 | 4000 | Most niche domain |
| C5 | cqadupstack/tex | 1e-6 | 4000 | TeX + midpoint LR |

### Gate criteria
- **PASS** (≥ +5% relative test MRR): proceed to Phase 1A + 1B using the winning dataset/LR
- **MARGINAL** (+1–5%): acceptable, proceed with best variant
- **FAIL** (≤ 0%): dataset also at ceiling — accept NFCorpus results as-is

**Estimated runtime**: ~15–20 min per variant on Kaggle T4. All 5 variants ≈ 1.5 hours.

In [ ]:
from pathlib import Path

REPO_URL    = "https://github.com/Abishek-Chakravarthy/fed-rag.git"
REPO_BRANCH = "diff-data-model-exp"

# ── Which variants to run ─────────────────────────────────────────────────
# To run a subset: VARIANTS = "C1,C4"
#VARIANTS = "C1,C2,C3,C4,C5"
VARIANTS = "C6,C7,C8"
# ── Paths ─────────────────────────────────────────────────────────────────
WORKSPACE   = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content")
REPO_DIR    = WORKSPACE / "fed-rag"
EXP_DIR     = REPO_DIR / "zz_coderuns" / "quality_aware_fedrag" / "diff_model_dataset_combos"
EXPORT_DIR  = WORKSPACE / "cqadupstack_diagnostic_exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"WORKSPACE  : {WORKSPACE}")
print(f"REPO_DIR   : {REPO_DIR}")
print(f"EXP_DIR    : {EXP_DIR}")
print(f"VARIANTS   : {VARIANTS}")

In [ ]:
import os
import shutil
import subprocess
import sys

# Clone repo (fresh every run to pick up any local pushes)
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, "--single-branch",
     REPO_URL, str(REPO_DIR)],
    check=True,
)

# Install dependencies — same pins as the existing federated notebooks
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"],
    check=True,
)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "protobuf>=4.25.3,<6",
        "accelerate",
        "datasets<3.0.0",
        "flwr==1.22.0",
        "pyarrow",
        "pydantic",
        "pydantic-settings",
        "transformers==4.48.0",
        "sentence-transformers==3.4.1",
        "peft",
        "matplotlib",
        "pandas",
        "tqdm",
    ],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR), "--no-deps"],
    check=True,
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
print("Clone + install complete")

In [ ]:
import importlib.metadata
import torch

print("torch version    :", torch.__version__)
print("protobuf version :", importlib.metadata.version("protobuf"))
print("cuda available   :", torch.cuda.is_available())
print("mps  available   :", torch.backends.mps.is_available())

if torch.cuda.is_available():
    print("gpu device       :", torch.cuda.get_device_name(0))
    subprocess.run(["nvidia-smi"])
elif torch.backends.mps.is_available():
    print("Using Apple Metal backend")
else:
    print("WARNING: No GPU detected. Switch the Kaggle runtime to GPU (T4 x2).")

## Run diagnostic

Each variant runs as a **separate subprocess** so GPU memory and Accelerator state are fully reset between variants. Live output is streamed to this cell.

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    "-u",                          # unbuffered stdout → live streaming
    "diagnostic_check.py",
    "--variants", VARIANTS,
]

print("Running:", " ".join(cmd))
print(f"EXP_DIR : {EXP_DIR}")
print("=" * 72)

process = subprocess.Popen(
    cmd,
    cwd=EXP_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="", flush=True)

process.wait()

if process.returncode != 0:
    print(f"\n⚠️  Process exited with code {process.returncode}. Check output above.")
else:
    print("\n✅ Diagnostic script exited cleanly.")

## Results and gate decision

In [ ]:
import json
import pandas as pd

summary_path = EXP_DIR / "diagnostic_results" / "diagnostic_summary.json"

if not summary_path.exists():
    print(f"No summary file found at {summary_path}. Did the script complete?")
else:
    with open(summary_path) as f:
        results = json.load(f)

    df = pd.DataFrame(results)[[
        "id", "dataset", "lr", "max_train",
        "pre_test_mrr", "post_test_mrr", "test_mrr_delta_pct",
        "pre_test_ndcg", "post_test_ndcg", "test_ndcg_delta_pct",
        "training_loss", "verdict",
    ]]
    df = df.sort_values("test_mrr_delta_pct", ascending=False)
    pd.set_option("display.float_format", "{:.6f}".format)
    display(df)

    # Gate summary
    print("\n" + "=" * 72)
    passing = [r for r in results if r["test_mrr_delta_pct"] >= 5.0]
    if passing:
        best = max(passing, key=lambda r: r["test_mrr_delta_pct"])
        print(f"✅ GATE PASSED")
        print(f"   Best variant : {best['id']} — {best['note']}")
        print(f"   Dataset      : {best['dataset']}")
        print(f"   LR to use    : {best['lr']:.0e}")
        print(f"   Δ test MRR   : {best['test_mrr_delta_pct']:+.2f}%")
        print()
        print("   Next steps:")
        print(f"   ► QA-FedAvg : set DATASET_NAME='{best['dataset']}', LEARNING_RATE={best['lr']:.0e} in federated_noisy_qa.py")
        print(f"   ► DAS-FedAvg: set TARGET_DATASET='{best['dataset']}', LEARNING_RATE={best['lr']:.0e} in federated_das.py")
    else:
        marginals = [r for r in results if r["test_mrr_delta_pct"] >= 1.0]
        if marginals:
            best = max(marginals, key=lambda r: r["test_mrr_delta_pct"])
            print(f"⚠️  GATE MARGINAL — best is {best['test_mrr_delta_pct']:+.2f}% (need ≥ 5%)")
            print(f"   Best variant: {best['id']} — {best['note']}")
            print(f"   Consider proceeding with this variant or trying more training epochs.")
        else:
            print(f"❌ GATE FAILED — no CQADupStack variant improved MRR.")
            print(f"   Both FiQA and CQADupStack are at ceiling for all-MiniLM-L6-v2.")
            print(f"   Recommendation: accept NFCorpus mechanism benchmark results as-is.")

## Export results

Copies all result JSON files and the summary to a zip for download.

In [ ]:
import shutil
import zipfile

diag_results_dir = EXP_DIR / "diagnostic_results"

if not diag_results_dir.exists():
    print("No diagnostic_results/ folder found — did the run complete?")
else:
    export_subdir = EXPORT_DIR / "diagnostic_results"
    if export_subdir.exists():
        shutil.rmtree(export_subdir)
    shutil.copytree(diag_results_dir, export_subdir)

    zip_path = EXPORT_DIR / "cqadupstack_diagnostic_results.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for file_path in export_subdir.rglob("*.json"):
            zf.write(file_path, arcname=file_path.name)

    print(f"Export folder : {export_subdir}")
    print(f"Zip file      : {zip_path}")
    print("\nFiles included:")
    for p in sorted(export_subdir.rglob("*.json")):
        print(f"  {p.name}")

## What to do with the results

**If GATE PASSED (≥ +5% test MRR on any CQADupStack variant):**

1. Note the winning dataset and LR from the summary above.
2. **QA-FedAvg** (`diff_model_dataset_combos/federated_noisy_qa.py`):
   ```python
   DATASET_NAME  = "cqadupstack/android"  # or /tex depending on winner
   LEARNING_RATE = <winner LR>
   ```
3. **DAS-FedAvg** (`diff_model_dataset_combos/federated_das.py`):
   ```python
   TARGET_DATASET = "cqadupstack/android"  # or /tex
   LEARNING_RATE  = <winner LR>
   ```
4. Run the federated experiments using the existing `single_alpha_kaggle_colab.ipynb` / `das_fedrag_single_seed_notebook.ipynb` pointing at `diff_model_dataset_combos/`.

**If GATE FAILED:**

Both FiQA and CQADupStack are at ceiling for `all-MiniLM-L6-v2`. Accept the NFCorpus results from the mechanism benchmark as-is — they demonstrate mechanism correctness, which is the primary contribution of both QA-FedAvg and DAS-FedAvg.